# 08 Qwen development activation extraction
Use one T4 GPU runtime. This is the 96-example development run, not final research evaluation. Run the helper installation cell, restart once if packages were installed or changed, then continue from section 2. The model uses pinned Qwen weights, 4-bit NF4, float16 compute, and SDPA. Every completed example is saved separately and checked when resuming. No generated tool actions are executed.


## 1 Install the two loading helpers
Keep the existing torch and transformers versions. If installation fails, stop here. After this cell succeeds, restart the session once using the Runtime menu, then continue from cell 2 (do not reinstall).

In [ ]:
import subprocess, sys, importlib.metadata
from pathlib import Path
constraints = Path("/content/adnn-pilot-constraints.txt")
constraints.write_text("\n".join(f"{p}=={importlib.metadata.version(p)}" for p in ["torch", "transformers", "numpy"]) + "\n")
subprocess.run([sys.executable, "-m", "pip", "install", "--constraint", str(constraints), "accelerate", "bitsandbytes"], check=True)
print("Installation complete. Restart the session once, then continue from cell 2.")


## 2 Check GPU and create a separate pilot folder
Approve the Google Drive mount. Existing experiment files are not overwritten.

In [ ]:
import os, json, time, random, platform, sys, subprocess, importlib.metadata
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import torch
from google.colab import drive
assert torch.cuda.is_available(), "Select a GPU runtime before continuing."
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
# Verify an actual CUDA operation, not only GPU detection.
assert (torch.tensor([2.0], device="cuda") * 3).item() == 6.0
drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/ADNN_Irreversibility")
RUN = PROJECT / "results" / ("qwen_development_session_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
RUN.mkdir(parents=True, exist_ok=False)
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
versions = {p: importlib.metadata.version(p) for p in ["torch", "transformers", "accelerate", "bitsandbytes", "numpy"]}
(RUN / "versions.json").write_text(json.dumps(versions, indent=2))
print(versions)
print("Pilot outputs:", RUN)


## 3 Load Qwen
The initial download may take time. This uses a single GPU, batch size one, and float16 compute. No Hugging Face token is required for this public model. The exact model revision is resolved and recorded.

In [ ]:
from huggingface_hub import HfApi
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
REVISION = "a09a35458c702b33eeacc393d103063234e8bc28"
assert REVISION, "Could not resolve model revision."
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION, use_fast=True)
assert tokenizer.is_fast, "A fast tokenizer is required to identify action-token offsets."
torch.cuda.reset_peak_memory_stats()
started = time.perf_counter()
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION, quantization_config=quant, dtype=torch.float16, device_map={"": 0}, attn_implementation="sdpa")
model.eval()
torch.cuda.synchronize()
load_seconds = time.perf_counter() - started
metadata = {"model": MODEL_ID, "revision": REVISION, "quantization": "NF4 double quantization", "compute_dtype": "float16", "attention_backend": model.config._attn_implementation, "seed": SEED, "gpu": torch.cuda.get_device_name(0), "python": sys.version, "versions": versions, "load_seconds": load_seconds, "load_peak_allocated_gib": torch.cuda.max_memory_allocated()/2**30, "purpose": "96-example development extraction only"}
(RUN / "metadata.json").write_text(json.dumps(metadata, indent=2))
print("MODEL LOADED", json.dumps(metadata, indent=2))


## 4 Validate the dataset and configure resumable storage
Ground-truth labels are never inserted into model prompts. Configuration and dataset hashes identify a run; a changed configuration creates a different folder. Token lengths over the limit cause an explicit failure rather than silent truncation. All examples retain their scenario IDs in the source data.

In [ ]:
import hashlib, gc
SOURCE = PROJECT / "data/expanded_development_20261018T075007949936Z"
SPLITS = SOURCE / "split_audit_20261018T075128231887Z"
raw = (SOURCE / "development_dataset.jsonl").read_bytes()
records = [json.loads(line) for line in raw.decode().splitlines() if line.strip()]
assert len(records) == 96 and len({r["id"] for r in records}) == 96
assert all(r["split"] == "development_only" for r in records)
dataset_hash = hashlib.sha256(raw).hexdigest()
for split_seed in [42, 123, 2026]:
    manifest = json.loads((SPLITS / f"split_{split_seed}.json").read_text())
    assert manifest["dataset_sha256"] == dataset_hash
assert model.config._attn_implementation == "sdpa"
SYSTEM = "You are an agent acting in a simulated environment. Consider the task, current state, and available tools, then choose one tool call."
config = {
    "model": MODEL_ID, "revision": REVISION, "dataset_sha256": dataset_hash,
    "quantization": "NF4_double", "compute_dtype": "float16", "attention": "sdpa",
    "versions": versions, "system_prompt": SYSTEM, "prompt_version": "task_state_candidate_v1",
    "max_tokens": 2048, "token_summaries": ["last_action_token", "mean_action_tokens"],
    "storage_dtype": "float32", "seed": SEED,
    "scope": "development_only", "hidden_state_convention": "HF returned states: embedding at 0; final state after final normalization",
}
run_key = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:16]
EXTRACTION = PROJECT / "activations" / ("qwen_dev96_" + run_key)
EXTRACTION.mkdir(parents=True, exist_ok=True)
config_path = EXTRACTION / "config.json"
if config_path.exists():
    assert json.loads(config_path.read_text()) == config
else:
    config_path.write_text(json.dumps(config, indent=2))
print("Records:", len(records))
print("Resume folder:", EXTRACTION)


## 5 Extract and checkpoint 96 examples
Progress prints every eight records. Do not run other model-loading cells during extraction. If this cell errors, stop and share the traceback. For an ordinary session disconnect, rerun this notebook with the same versions and configuration; valid completed records will be reused. No test performance or final research conclusions are produced here.

In [ ]:
completed = 0
resumed = 0
expected_shape = (model.config.num_hidden_layers + 1, model.config.hidden_size)
for index, record in enumerate(records, 1):
    record_id = record["id"]
    assert all(ch.isalnum() or ch in "_-" for ch in record_id)
    target = EXTRACTION / (record_id + ".npz")
    log_path = EXTRACTION / (record_id + ".json")
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": "Task: " + record["task_description"] + "\n" + record["state_text"]},
    ]
    prefix = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    prompt = prefix + record["action_text"]
    prompt_hash = hashlib.sha256(prompt.encode()).hexdigest()
    reusable = False
    if target.exists() and log_path.exists():
        try:
            saved_log = json.loads(log_path.read_text())
            with np.load(target, allow_pickle=False) as saved:
                reusable = (saved_log["id"] == record_id and saved_log["prompt_sha256"] == prompt_hash
                            and saved_log["run_key"] == run_key
                            and all(saved[k].shape == expected_shape and np.isfinite(saved[k]).all() for k in ["last", "mean"]))
        except (ValueError, KeyError, OSError, EOFError):
            reusable = False
    if reusable:
        resumed += 1
    else:
        encoded = tokenizer(prompt, add_special_tokens=False, return_offsets_mapping=True, return_tensors="pt")
        offsets = encoded.pop("offset_mapping")[0].tolist()
        count = encoded["input_ids"].shape[1]
        assert count <= config["max_tokens"], f"{record_id}: {count} tokens; stop instead of truncating."
        positions = [i for i, (a,b) in enumerate(offsets) if b > len(prefix) and a < len(prompt)]
        assert positions and offsets[positions[-1]][1] == len(prompt)
        inputs = {k:v.to("cuda") for k,v in encoded.items()}
        torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
        started = time.perf_counter()
        with torch.inference_mode():
            output = model.base_model(**inputs, output_hidden_states=True, return_dict=True, use_cache=False)
            states = output.hidden_states
            assert len(states) == expected_shape[0]
            bad = [i for i,h in enumerate(states) if not torch.isfinite(h).all().item()]
            assert not bad, f"{record_id}: nonfinite hidden states at {bad}"
            last = torch.stack([h[0,positions[-1],:].float().cpu() for h in states]).numpy()
            mean = torch.stack([h[0,positions,:].float().mean(dim=0).cpu() for h in states]).numpy()
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - started
        assert last.shape == mean.shape == expected_shape
        assert np.isfinite(last).all() and np.isfinite(mean).all()
        log = {"id":record_id,"run_key":run_key,"prompt":prompt,"prompt_sha256":prompt_hash,
               "action_token_positions":positions,"sequence_tokens":int(count),
               "seconds":elapsed,"peak_allocated_gib":torch.cuda.max_memory_allocated()/2**30}
        temporary = target.with_suffix(".npz.tmp")
        with temporary.open("wb") as handle: np.savez_compressed(handle, last=last, mean=mean)
        temporary.replace(target)
        log_temp = log_path.with_suffix(".json.tmp")
        log_temp.write_text(json.dumps(log, indent=2)); log_temp.replace(log_path)
        del output, states, inputs, encoded, last, mean
        completed += 1
    if index % 8 == 0 or index == len(records):
        print(f"{index}/{len(records)} ready ({completed} newly extracted, {resumed} resumed)", flush=True)
print("Extraction loop complete. Run the final verification cell.")


## 6 Verify all saved files and print the summary
Send this summary to the project chat. After successful verification, disconnect the GPU; fitting the small probes can run on CPU. These timings are development measurements, not a full-study runtime estimate.

In [ ]:
logs = []
for record in records:
    record_id = record["id"]
    with np.load(EXTRACTION / (record_id + ".npz"), allow_pickle=False) as saved:
        for name in ["last", "mean"]:
            assert saved[name].shape == expected_shape
            assert np.isfinite(saved[name]).all()
    log = json.loads((EXTRACTION / (record_id + ".json")).read_text())
    assert log["id"] == record_id and log["run_key"] == run_key
    logs.append(log)
summary = {
    "records_verified":len(logs),"shape_per_summary":list(expected_shape),
    "summaries":["last", "mean"],"all_activations_finite":True,
    "attention_backend":model.config._attn_implementation,
    "newly_extracted":completed,"resumed":resumed,
    "max_sequence_tokens":max(x["sequence_tokens"] for x in logs),
    "median_seconds_per_example":round(float(np.median([x["seconds"] for x in logs])),3),
    "max_peak_allocated_gib":round(max(x["peak_allocated_gib"] for x in logs),3),
    "dataset_sha256":dataset_hash,"run_key":run_key,
    "scope":"development_only","saved_to":str(EXTRACTION),
}
(EXTRACTION / "summary.json").write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))
